In [ ]:
import math_toolkit
math_toolkit.notebook.activate()

# figure

`figure(...)` creates a durable plotting container so you can display once, then add curves, fields, domains, and parametric plots to the same live figure.

Primary forms:

```python
fig = figure("name")
fig.show()
fig.plot(expr, x)
```

```python
fig = figure("name")
fig.show()
with fig:
    plot(expr, x)
```

In [ ]:
x, a = symbols("x a")

fig = figure("figure-quick-payoff")
fig.show()
fig.parameters({a: 1.0})
fig.plot(sin(a * x), (x, -2 * pi, 2 * pi), label="sin(a x)")

## Core behavior

A figure owns the plots, controls, display widget, and view settings for one plotting surface. The normal notebook rhythm is to create the figure, show it, and then add or update plots through the figure handle.

`figure("name")` returns the same named figure each time. That makes rerun-friendly notebook cells: you can recreate the handle in a later cell and still target the same plotting surface.

`figure()` with no name creates a fresh unnamed figure. Use an unnamed figure when you want a scratch plotting surface that does not need to be recovered by name later.

In [ ]:
named = figure("phase-space")
same_named = figure("phase-space")

md("""The repeated named lookup returns the same object: {{ named is same_named }}.""")

## Common patterns

**Show first, then add plots.** Calling `fig.show()` creates a visible live display. Later `fig.plot(...)` calls update that display instead of making a separate plotting surface.

**Use `with fig:` for grouped plotting cells.** Inside the context block, top-level plot commands route to `fig`. This is useful when you want compact mathematical code but still want an explicit figure owner.

**Use figure methods for the most explicit form.** `fig.plot(...)`, `fig.list_plot(...)`, `fig.temperature_plot(...)`, `fig.contour_plot(...)`, `fig.domain_plot(...)`, and `fig.parametric_plot(...)` all add plots directly to `fig`. See the linked plot-kind pages for discrete points, heatmaps, contours, domains, and parametric curves.

**Keep plot identity separate from figure identity.** The figure name identifies the plotting surface; a plot `name=` identifies one trace or field inside that surface.

In [ ]:
fig = figure("context-routing")
fig.show()

with fig:
    plot(sin(x), (x, -pi, pi), name="sine", label="sin x")
    plot(cos(x), (x, -pi, pi), name="cosine", label="cos x")

## Options and Details

**Display options.** `fig.show()` returns a display generation and accepts display-oriented keyword arguments such as `backend=`, `policy=`, and `new=`. The default `new=True` creates a fresh display generation. Use `new=False` when you want to ask the current active generation to display again.

**Widget access.** After a figure has been shown, `fig.widget` returns the active layout widget, `fig.figure_widget` returns the underlying Plotly figure widget, and `fig.layout` returns the active layout shell.

**Custom layouts.** Pass `layout=SomeLayoutClass` to `figure(...)`, or call `fig.set_layout(SomeLayoutClass)`, when you want future display generations to arrange the plot, controls, and status area differently. The layout value is a class, not a prebuilt widget instance.

**View controls.** A figure owns Cartesian views. Use `fig.view("name")` to create or retrieve a named view, assign `fig.view.range = {"x": (xmin, xmax), "y": (ymin, ymax)}` to set visible axis ranges, assign `fig.view.home_range` to set reset ranges, and call `fig.view.reset()` to return to the home range.

**Control maintenance.** Parameter sliders are normally reconciled automatically. `fig.rebuild_controls()` and `fig.reconcile_controls()` are available when custom display code needs to refresh controls explicitly; `fig.sync_controls(...)` applies an existing slider value snapshot.

In [ ]:
fig = figure("windowed-view")
fig.show()
fig.plot(sin(x) / x, (x, -12, 12), name="sinc", label="sin(x)/x")
fig.view.range = {"x": (-8, 8), "y": (-0.4, 1.2)}

## Parameter animation

Figure parameters are animatable by default in the anywidget frontend. Add animation metadata next to the ordinary slider metadata with `animated`, `animation_mode`, `animation_rate_hz`, and `animation_speed`.

Use the parameter-scoped handle for playback commands: `fig.parameter[a].animate.start()`, `.stop()`, `.toggle()`, and `.enabled`. The plural spelling also works: `fig.parameters[a].animate.start()`.

`animation_rate_hz` controls the Python recomputation cadence and accepts `60`, `30`, `20`, `10`, `5`, `2`, or `1`. `animation_speed` controls motion in parameter units per second. The default speed is step-derived, so `"default"` means `2 * step` per second. Supported modes are `"forward"`, `"bounce"`, and `"wrap"`.

In [ ]:
anim_fig = figure("figure-parameter-animation")
anim_fig.show(backend="anywidget")
anim_fig.parameters({
    a: {
        "value": 1.0,
        "min": 0.25,
        "max": 3.0,
        "step": 0.05,
        "animation_mode": "bounce",
        "animation_rate_hz": 20,
        "animation_speed": "default",
    }
})
anim_fig.plot(sin(a * x), (x, -2 * pi, 2 * pi), label="sin(a x)")

anim_fig.parameter[a].animate.enabled = True
# In a live notebook, click the play button after the slider or call:
# anim_fig.parameter[a].animate.start()
anim_fig.parameter[a].animate.stop()

## Authored info

`fig.info(...)` adds a Markdown card to the figure sidebar. Strings are literal Markdown, symbolic fragments create the same kind of sliders as plot parameters, and callables receive the figure for small computed diagnostics.

Named info cards update in place, so repeated cells can refresh a note without appending duplicates.

In [ ]:
info_fig = figure("figure-info-card")
info_fig.show()
info_fig.parameters({a: 1.0})
info_fig.plot(a * sin(x), (x, -2 * pi, 2 * pi), name="scaled")
info_fig.info(
    "Current scale: ",
    a,
    "\n\nPeak at the origin: ",
    lambda fig: fig.params[a]["value"],
    name="scale-summary",
    title="Live info",
)

## Semantics

A figure is a durable owner, not just a one-time display call. Adding a plot mutates that figure's plot collection; showing a figure creates or reuses a display generation for the same collection.

Named figures are reusable by name. Unnamed figures are fresh objects. Passing an existing figure handle to `figure(existing_fig)` returns that same handle.

The `with fig:` form temporarily routes top-level plotting calls and notebook output into the figure for the body of the block, then restores the previous routing when the block exits.

## Examples and Applications

### Example: build one figure from several plot kinds

One figure can hold compatible curve and region plots. Use names when you plan to update or retrieve individual plots later.

In [ ]:
fig = figure("mixed-geometry")
fig.show()

fig.domain_plot(x**2 + y**2 < 1, (x, -1.5, 1.5), (y, -1.5, 1.5), name="disk")
fig.parametric_plot((cos(x), sin(x)), (x, 0, 2 * pi), name="unit-circle")

## Pitfalls

**Creating a figure does not display it.** Use `fig.show()` when you want the live widget to appear before or after adding plots.

**A figure name is not a plot name.** `figure("demo")` selects the plotting surface. `fig.plot(..., name="curve")` selects a plot inside that surface.

**A named figure is reused.** If a later cell calls `figure("demo")`, it gets the existing named figure. Choose a new name or use `figure()` when you want a clean independent surface.

**Top-level `plot(...)` needs a route.** Prefer `fig.plot(...)` or `with fig:` in documentation and reusable notebooks so the target figure is visible in the code.

## See also

- [plot](plot.ipynb) for ordinary scalar curves and plot handles.
- [list_plot](list_plot.ipynb) for discrete point sets.
- [temperature_plot](temperature_plot.ipynb) for heatmap scalar fields.
- [contour_plot](contour_plot.ipynb) for contour scalar fields.
- [domain_plot](domain_plot.ipynb) for Boolean and signed domains.
- [parametric_plot](parametric_plot.ipynb) for two-coordinate parametric curves.
- [Num](Num.ipynb) for the symbolic-to-numeric boundary used by sampled plots.
- [Expression](Expression.ipynb) for symbolic expressions that can be plotted after activation.
- [Hamiltonian dynamics worksheet](../worksheets/hamiltonian_dynamics.ipynb) for a longer mathematical worksheet using plotting in context.